In [1]:
import pandas as pd
import numpy as np
import os
from pathlib import Path

print("Temporal behavioral analysis started.")

Temporal behavioral analysis started.


In [2]:
fake_tweets_path = Path(
    "../data/raw/datasets_full.csv/fake_followers.csv/fake_followers.csv/tweets.csv"
)

print("Exists:", fake_tweets_path.exists())
print("Path:", fake_tweets_path)

Exists: True
Path: ..\data\raw\datasets_full.csv\fake_followers.csv\fake_followers.csv\tweets.csv


In [3]:
fake_tweets = pd.read_csv(
    fake_tweets_path,
    encoding="latin1"
)

print("Shape:", fake_tweets.shape)
print("Columns:")
print(fake_tweets.columns.tolist())

fake_tweets.head()

Shape: (196027, 23)
Columns:
['created_at', 'id', 'text', 'source', 'user_id', 'truncated', 'in_reply_to_status_id', 'in_reply_to_user_id', 'in_reply_to_screen_name', 'retweeted_status_id', 'geo', 'place', 'contributors', 'retweet_count', 'reply_count', 'favorite_count', 'favorited', 'retweeted', 'possibly_sensitive', 'num_hashtags', 'num_urls', 'num_mentions', 'timestamp']


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\2785169419.py:1: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  fake_tweets = pd.read_csv(


,created_at,id,text,source,user_id,truncated,in_reply_to_status_id,in_reply_to_user_id,in_reply_to_screen_name,retweeted_status_id,...,retweet_count,reply_count,favorite_count,favorited,retweeted,possibly_sensitive,num_hashtags,num_urls,num_mentions,timestamp
0,Sat Apr 20 13:19:19 +0000 2013,325599560959393793,https://t.co/iocNIgHxXH. @LovesOfaLDNgirl her...,"<a href=""http://twitter.com/download/iphone"" r...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,1,2013-04-20 15:19:19
1,Tue Apr 16 19:31:39 +0000 2013,324243711443730434,Well done hubby @Allan_76 http://t.co/AaeTwLucUG,"<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,1,2013-04-16 21:31:39
2,Tue Apr 16 17:38:06 +0000 2013,324215137055670274,Two years with my lovely husband - thank you f...,"<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,1,2013-04-16 19:38:06
3,Sun Apr 14 15:33:00 +0000 2013,323458877003792386,Sorry bunny about your ears but I was hungry.....,"<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,0,0,NaN,NaN,NaN,0,1,0,2013-04-14 17:33:00
4,Fri Apr 12 15:37:59 +0000 2013,322735354148945920,"Small man, big drink @Allan_76 http://t.co/4NU...","<a href=""http://instagram.com"" rel=""nofollow"">...",10935572,NaN,0,0,NaN,NaN,...,0,1,0,NaN,NaN,NaN,0,1,1,2013-04-12 17:37:59


In [4]:
# Keep only the columns we need
temp = fake_tweets[["user_id", "created_at"]].copy()

# Convert timestamps
temp["created_at"] = pd.to_datetime(
    temp["created_at"],
    errors="coerce"
)

# Remove invalid timestamps
temp = temp.dropna(subset=["user_id", "created_at"])

# Sort by account and time
temp = temp.sort_values(["user_id", "created_at"])

# Calculate time difference between consecutive tweets
temp["intertweet_seconds"] = (
    temp.groupby("user_id")["created_at"]
        .diff()
        .dt.total_seconds()
)

# Calculate mean interval for each account
fake_mean_intertweet = (
    temp.groupby("user_id")["intertweet_seconds"]
        .mean()
        .reset_index(name="mean_intertweet_seconds")
)

print("Accounts:", len(fake_mean_intertweet))
fake_mean_intertweet.head()

C:\Users\namit\AppData\Local\Temp\ipykernel_1832\998346039.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Accounts: 3202


,user_id,mean_intertweet_seconds
0,10935572,3.079435e+05
1,16119337,4.881397e+04
2,16753788,4.717938e+06
3,17640121,6.130340e+06
4,17656600,4.257927e+05


In [5]:
fake_mean_intertweet["mean_intertweet_seconds"].describe()

count    3.088000e+03
mean     1.256946e+06
std      4.625927e+06
min      2.000000e+00
25%      3.341127e+05
50%      4.725721e+05
75%      8.545844e+05
max      1.036911e+08
Name: mean_intertweet_seconds, dtype: float64

In [6]:
print("Number of accounts:", len(fake_mean_intertweet))

print("\nMissing values:")
print(fake_mean_intertweet["mean_intertweet_seconds"].isna().sum())

print("\nDescriptive statistics:")
print(fake_mean_intertweet["mean_intertweet_seconds"].describe())

print("\nLargest 10 mean intervals:")
print(
    fake_mean_intertweet
    .sort_values("mean_intertweet_seconds", ascending=False)
    .head(10)
)

Number of accounts: 3202

Missing values:
114

Descriptive statistics:
count    3.088000e+03
mean     1.256946e+06
std      4.625927e+06
min      2.000000e+00
25%      3.341127e+05
50%      4.725721e+05
75%      8.545844e+05
max      1.036911e+08
Name: mean_intertweet_seconds, dtype: float64

Largest 10 mean intervals:
       user_id  mean_intertweet_seconds
57    56163699              103691143.0
194  119926900               97241502.0
330  190265169               80967303.0
147   96871607               72371346.0
244  142150030               64757158.0
441  254839933               59504813.0
531  328246965               57596892.0
120   82122078               49204855.5
286  163142123               44306359.0
502  303469076               36854702.0


In [7]:
fake_mean_intertweet["mean_intertweet_hours"] = (
    fake_mean_intertweet["mean_intertweet_seconds"] / 3600
)

print(
    fake_mean_intertweet["mean_intertweet_hours"].describe()
)

count     3088.000000
mean       349.151675
std       1284.979825
min          0.000556
25%         92.809093
50%        131.270016
75%        237.384552
max      28803.095278
Name: mean_intertweet_hours, dtype: float64


In [8]:
tweet_counts = (
    temp.groupby("user_id")
    .size()
    .reset_index(name="tweet_count")
)

fake_mean_intertweet = fake_mean_intertweet.merge(
    tweet_counts,
    on="user_id",
    how="left"
)

print(fake_mean_intertweet["tweet_count"].describe())

count    3202.000000
mean       61.220175
std       232.266613
min         1.000000
25%        17.000000
50%        23.000000
75%        40.000000
max      3383.000000
Name: tweet_count, dtype: float64


In [9]:
print(
    fake_mean_intertweet[
        ["user_id", "tweet_count", "mean_intertweet_seconds"]
    ].head(10)
)

    user_id  tweet_count  mean_intertweet_seconds
0  10935572          556             3.079435e+05
1  16119337          355             4.881397e+04
2  16753788           30             4.717938e+06
3  17640121           20             6.130340e+06
4  17656600          306             4.257927e+05
5  19230427           96             6.695184e+05
6  19951698          193             1.416404e+05
7  20731719          436             3.071765e+05
8  21196037           78             1.568908e+06
9  21375100          218             5.812848e+05


In [10]:
median_intertweet = (
    temp.groupby("user_id")["intertweet_seconds"]
        .median()
        .reset_index(name="median_intertweet_seconds")
)

print("Accounts:", len(median_intertweet))

print("\nStatistics:")
print(median_intertweet["median_intertweet_seconds"].describe())

print("\nFirst 10 accounts:")
print(median_intertweet.head(10))

Accounts: 3202

Statistics:
count    3.088000e+03
mean     6.974866e+05
std      4.272561e+06
min      2.000000e+00
25%      1.762214e+05
50%      2.694390e+05
75%      3.807890e+05
max      1.036911e+08
Name: median_intertweet_seconds, dtype: float64

First 10 accounts:
    user_id  median_intertweet_seconds
0  10935572                    24136.0
1  16119337                       55.5
2  16753788                  1798473.0
3  17640121                  1517379.0
4  17656600                    35335.0
5  19230427                   172683.0
6  19951698                       20.0
7  20731719                       11.0
8  21196037                     7696.0
9  21375100                    69498.0


In [11]:
median_intertweet["median_intertweet_hours"] = (
    median_intertweet["median_intertweet_seconds"] / 3600
)

print(median_intertweet["median_intertweet_hours"].describe())

count     3088.000000
mean       193.746291
std       1186.822497
min          0.000556
25%         48.950382
50%         74.844167
75%        105.774722
max      28803.095278
Name: median_intertweet_hours, dtype: float64


In [12]:
temp["intertweet_seconds"]

553              NaN
552       36893348.0
551           5911.0
550         192524.0
549          85088.0
             ...    
196022           NaN
196024           NaN
196023     1163404.0
196026           NaN
196025     1164225.0
Name: intertweet_seconds, Length: 196027, dtype: float64

In [13]:
intertweet_std = (
    temp.groupby("user_id")["intertweet_seconds"]
        .std()
        .reset_index(name="intertweet_std_seconds")
)

print("Accounts:", len(intertweet_std))

print("\nStatistics:")
print(intertweet_std["intertweet_std_seconds"].describe())

Accounts: 3202

Statistics:
count    2.998000e+03
mean     1.754492e+06
std      4.639735e+06
min      0.000000e+00
25%      3.123197e+05
50%      6.436917e+05
75%      1.233665e+06
max      6.952952e+07
Name: intertweet_std_seconds, dtype: float64


In [14]:
intertweet_stats = (
    temp.groupby("user_id")["intertweet_seconds"]
        .agg(
            mean_intertweet_seconds="mean",
            std_intertweet_seconds="std"
        )
        .reset_index()
)

intertweet_stats["intertweet_cv"] = (
    intertweet_stats["std_intertweet_seconds"] /
    intertweet_stats["mean_intertweet_seconds"]
)

print(intertweet_stats.head(10))

print("\nCV statistics:")
print(intertweet_stats["intertweet_cv"].describe())

    user_id  mean_intertweet_seconds  std_intertweet_seconds  intertweet_cv
0  10935572             3.079435e+05            2.074548e+06       6.736780
1  16119337             4.881397e+04            5.457522e+05      11.180248
2  16753788             4.717938e+06            5.885562e+06       1.247486
3  17640121             6.130340e+06            8.649692e+06       1.410965
4  17656600             4.257927e+05            1.317960e+06       3.095309
5  19230427             6.695184e+05            1.408532e+06       2.103799
6  19951698             1.416404e+05            7.618125e+05       5.378496
7  20731719             3.071765e+05            4.407376e+06      14.348022
8  21196037             1.568908e+06            6.758710e+06       4.307907
9  21375100             5.812848e+05            2.142826e+06       3.686362

CV statistics:
count    2998.000000
mean        1.640041
std         1.829862
min         0.000000
25%         0.905957
50%         1.227131
75%         1.577075
m

In [15]:
intertweet_stats["burstiness"] = (
    intertweet_stats["std_intertweet_seconds"]
    - intertweet_stats["mean_intertweet_seconds"]
) / (
    intertweet_stats["std_intertweet_seconds"]
    + intertweet_stats["mean_intertweet_seconds"]
)

In [16]:
print("Burstiness statistics:")
print(intertweet_stats["burstiness"].describe())

Burstiness statistics:
count    2998.000000
mean        0.117217
std         0.242291
min        -1.000000
25%        -0.049342
50%         0.101984
75%         0.223926
max         0.942490
Name: burstiness, dtype: float64


In [17]:
print("\nBurstiness range:")
print(
    intertweet_stats["burstiness"].min(),
    "to",
    intertweet_stats["burstiness"].max()
)


Burstiness range:
-1.0 to 0.9424900012270045


In [18]:
print("\nFirst 10 accounts:")
print(
    intertweet_stats[
        ["user_id",
         "mean_intertweet_seconds",
         "std_intertweet_seconds",
         "intertweet_cv",
         "burstiness"]
    ].head(10)
)


First 10 accounts:
    user_id  mean_intertweet_seconds  std_intertweet_seconds  intertweet_cv  \
0  10935572             3.079435e+05            2.074548e+06       6.736780   
1  16119337             4.881397e+04            5.457522e+05      11.180248   
2  16753788             4.717938e+06            5.885562e+06       1.247486   
3  17640121             6.130340e+06            8.649692e+06       1.410965   
4  17656600             4.257927e+05            1.317960e+06       3.095309   
5  19230427             6.695184e+05            1.408532e+06       2.103799   
6  19951698             1.416404e+05            7.618125e+05       5.378496   
7  20731719             3.071765e+05            4.407376e+06      14.348022   
8  21196037             1.568908e+06            6.758710e+06       4.307907   
9  21375100             5.812848e+05            2.142826e+06       3.686362   

   burstiness  
0    0.741495  
1    0.835800  
2    0.110117  
3    0.170456  
4    0.511636  
5    0.355628 

In [19]:
temp["posting_hour"] = temp["created_at"].dt.hour

print(temp[["user_id", "created_at", "posting_hour"]].head(10))

      user_id                created_at  posting_hour
553  10935572 2007-12-07 13:57:21+00:00            13
552  10935572 2009-02-06 14:06:29+00:00            14
551  10935572 2009-02-06 15:45:00+00:00            15
550  10935572 2009-02-08 21:13:44+00:00            21
549  10935572 2009-02-09 20:51:52+00:00            20
548  10935572 2009-02-14 19:12:33+00:00            19
547  10935572 2009-02-15 21:31:16+00:00            21
546  10935572 2009-02-28 20:30:01+00:00            20
545  10935572 2009-06-26 16:20:09+00:00            16
544  10935572 2009-08-12 17:23:59+00:00            17


In [20]:
def calculate_hour_entropy(hours):
    counts = hours.value_counts(normalize=True)
    return -(counts * np.log2(counts)).sum()

hour_entropy = (
    temp.groupby("user_id")["posting_hour"]
        .apply(calculate_hour_entropy)
        .reset_index(name="posting_time_entropy")
)

print("Accounts:", len(hour_entropy))
print("\nEntropy statistics:")
print(hour_entropy["posting_time_entropy"].describe())

Accounts: 3202

Entropy statistics:
count    3202.000000
mean        3.342091
std         1.023852
min        -0.000000
25%         3.251629
50%         3.628785
75%         3.960362
max         4.514577
Name: posting_time_entropy, dtype: float64


In [21]:
print(
    "Entropy range:",
    hour_entropy["posting_time_entropy"].min(),
    "to",
    hour_entropy["posting_time_entropy"].max()
)

Entropy range: -0.0 to 4.514577483453128


In [22]:
active_hour_count = (
    temp.groupby("user_id")["posting_hour"]
        .nunique()
        .reset_index(name="active_hour_count")
)

print("Accounts:", len(active_hour_count))

print("\nStatistics:")
print(active_hour_count["active_hour_count"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean       13.791380
std         5.964716
min         1.000000
25%        11.000000
50%        14.000000
75%        18.000000
max        24.000000
Name: active_hour_count, dtype: float64


In [23]:
print("\nValue counts:")
print(
    active_hour_count["active_hour_count"]
    .value_counts()
    .sort_index()
)


Value counts:
active_hour_count
1     131
2     117
3      73
4      47
5      36
6      30
7      36
8      48
9      84
10    119
11    199
12    264
13    288
14    242
15    244
16    188
17    162
18    137
19    131
20    144
21    154
22    150
23    106
24     72
Name: count, dtype: int64


In [24]:
fake_temporal_features = (
    fake_mean_intertweet
    .merge(
        intertweet_stats[
            ["user_id", "intertweet_cv", "burstiness"]
        ],
        on="user_id",
        how="left"
    )
    .merge(
        median_intertweet[
            ["user_id", "median_intertweet_seconds"]
        ],
        on="user_id",
        how="left"
    )
    .merge(
        hour_entropy,
        on="user_id",
        how="left"
    )
    .merge(
        active_hour_count,
        on="user_id",
        how="left"
    )
)

print("Shape:", fake_temporal_features.shape)
print("\nColumns:")
print(fake_temporal_features.columns.tolist())

fake_temporal_features.head()

Shape: (3202, 9)

Columns:
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count']


,user_id,mean_intertweet_seconds,mean_intertweet_hours,tweet_count,intertweet_cv,burstiness,median_intertweet_seconds,posting_time_entropy,active_hour_count
0,10935572,3.079435e+05,85.539871,556,6.736780,0.741495,24136.0,4.051593,21
1,16119337,4.881397e+04,13.559435,355,11.180248,0.835800,55.5,3.547240,19
2,16753788,4.717938e+06,1310.538199,30,1.247486,0.110117,1798473.0,3.694740,15
3,17640121,6.130340e+06,1702.872266,20,1.410965,0.170456,1517379.0,3.246439,11
4,17656600,4.257927e+05,118.275749,306,3.095309,0.511636,35335.0,4.169805,24


In [25]:
temp["is_night"] = temp["posting_hour"].between(0, 5)

night_activity_ratio = (
    temp.groupby("user_id")["is_night"]
        .mean()
        .reset_index(name="night_activity_ratio")
)

print("Accounts:", len(night_activity_ratio))

print("\nStatistics:")
print(night_activity_ratio["night_activity_ratio"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean        0.215119
std         0.162782
min         0.000000
25%         0.125000
50%         0.193265
75%         0.268657
max         1.000000
Name: night_activity_ratio, dtype: float64


In [26]:
print(
    "Range:",
    night_activity_ratio["night_activity_ratio"].min(),
    "to",
    night_activity_ratio["night_activity_ratio"].max()
)

print("\nFirst 10:")
print(night_activity_ratio.head(10))

Range: 0.0 to 1.0

First 10:
    user_id  night_activity_ratio
0  10935572              0.026978
1  16119337              0.473239
2  16753788              0.266667
3  17640121              0.400000
4  17656600              0.333333
5  19230427              0.010417
6  19951698              0.046632
7  20731719              0.236239
8  21196037              0.217949
9  21375100              0.495413


In [27]:
fake_temporal_features = fake_temporal_features.merge(
    night_activity_ratio,
    on="user_id",
    how="left"
)

print("Shape:", fake_temporal_features.shape)
print(fake_temporal_features.columns.tolist())

Shape: (3202, 10)
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio']


In [28]:
temp["weekday"] = temp["created_at"].dt.weekday

temp["is_weekend"] = temp["weekday"].isin([5, 6])

weekend_activity_ratio = (
    temp.groupby("user_id")["is_weekend"]
        .mean()
        .reset_index(name="weekend_activity_ratio")
)

print("Accounts:", len(weekend_activity_ratio))

print("\nStatistics:")
print(weekend_activity_ratio["weekend_activity_ratio"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean        0.264214
std         0.145217
min         0.000000
25%         0.191255
50%         0.263158
75%         0.333333
max         1.000000
Name: weekend_activity_ratio, dtype: float64


In [29]:
print(
    "Range:",
    weekend_activity_ratio["weekend_activity_ratio"].min(),
    "to",
    weekend_activity_ratio["weekend_activity_ratio"].max()
)

print("\nFirst 10:")
print(weekend_activity_ratio.head(10))

Range: 0.0 to 1.0

First 10:
    user_id  weekend_activity_ratio
0  10935572                0.320144
1  16119337                0.016901
2  16753788                0.166667
3  17640121                0.250000
4  17656600                0.199346
5  19230427                0.312500
6  19951698                0.626943
7  20731719                0.332569
8  21196037                0.282051
9  21375100                0.266055


In [30]:
fake_temporal_features = fake_temporal_features.merge(
    weekend_activity_ratio,
    on="user_id",
    how="left"
)

print("Shape:", fake_temporal_features.shape)
print(fake_temporal_features.columns.tolist())

Shape: (3202, 11)
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio', 'weekend_activity_ratio']


In [31]:
temp["date"] = temp["created_at"].dt.date

daily_counts = (
    temp.groupby(["user_id", "date"])
        .size()
        .reset_index(name="daily_tweet_count")
)

print("Account-day records:", len(daily_counts))

print("\nDaily tweet count statistics:")
print(daily_counts["daily_tweet_count"].describe())

Account-day records: 87942

Daily tweet count statistics:
count    87942.000000
mean         2.229049
std          8.195446
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max        579.000000
Name: daily_tweet_count, dtype: float64


In [32]:
daily_activity_stats = (
    daily_counts.groupby("user_id")["daily_tweet_count"]
        .agg(
            mean_daily_tweets="mean",
            std_daily_tweets="std"
        )
        .reset_index()
)

daily_activity_stats["daily_activity_cv"] = (
    daily_activity_stats["std_daily_tweets"] /
    daily_activity_stats["mean_daily_tweets"]
)

print("Accounts:", len(daily_activity_stats))

print("\nDaily activity CV statistics:")
print(daily_activity_stats["daily_activity_cv"].describe())

Accounts: 3202

Daily activity CV statistics:
count    3061.000000
mean        0.396657
std         0.377953
min         0.000000
25%         0.235294
50%         0.317744
75%         0.450533
max         4.555788
Name: daily_activity_cv, dtype: float64


In [33]:
print(
    daily_activity_stats[
        ["user_id", "mean_daily_tweets",
         "std_daily_tweets", "daily_activity_cv"]
    ].head(10)
)

    user_id  mean_daily_tweets  std_daily_tweets  daily_activity_cv
0  10935572           2.206349          1.890709           0.856940
1  16119337          39.444444         76.313680           1.934713
2  16753788           1.304348          1.258960           0.965203
3  17640121           1.176471          0.392953           0.334010
4  17656600           1.987013          1.652833           0.831818
5  19230427           1.411765          0.737792           0.522603
6  19951698           6.031250         14.894217           2.469507
7  20731719          24.222222         21.821618           0.900892
8  21196037           2.516129          2.079599           0.826507
9  21375100           1.772358          1.304551           0.736054


In [34]:
top_hour_concentration = (
    temp.groupby("user_id")["posting_hour"]
        .value_counts(normalize=True)
        .groupby(level=0)
        .max()
        .reset_index(name="top_hour_concentration")
)

print("Accounts:", len(top_hour_concentration))

print("\nStatistics:")
print(top_hour_concentration["top_hour_concentration"].describe())

Accounts: 3202

Statistics:
count    3202.000000
mean        0.217443
std         0.194591
min         0.067086
25%         0.121212
50%         0.153846
75%         0.210526
max         1.000000
Name: top_hour_concentration, dtype: float64


In [35]:
print(
    "Range:",
    top_hour_concentration["top_hour_concentration"].min(),
    "to",
    top_hour_concentration["top_hour_concentration"].max()
)

print("\nFirst 10:")
print(top_hour_concentration.head(10))

Range: 0.06708595387840671 to 1.0

First 10:
    user_id  top_hour_concentration
0  10935572                0.129496
1  16119337                0.191549
2  16753788                0.166667
3  17640121                0.200000
4  17656600                0.104575
5  19230427                0.156250
6  19951698                0.466321
7  20731719                0.126147
8  21196037                0.205128
9  21375100                0.155963


In [36]:
fake_temporal_features = fake_temporal_features.merge(
    top_hour_concentration,
    on="user_id",
    how="left"
)

print("Shape:", fake_temporal_features.shape)
print(fake_temporal_features.columns.tolist())

Shape: (3202, 12)
['user_id', 'mean_intertweet_seconds', 'mean_intertweet_hours', 'tweet_count', 'intertweet_cv', 'burstiness', 'median_intertweet_seconds', 'posting_time_entropy', 'active_hour_count', 'night_activity_ratio', 'weekend_activity_ratio', 'top_hour_concentration']


In [37]:
import pandas as pd
import numpy as np
from pathlib import Path

In [58]:
def build_temporal_features(tweets_df):

    # ---------------------------------------------------------
    # 1. Select required columns
    # ---------------------------------------------------------
    temp = tweets_df[["user_id", "created_at"]].copy()

    # ---------------------------------------------------------
    # 2. Clean timestamp values
    # ---------------------------------------------------------
    created_raw = temp["created_at"].astype(str).str.strip()

    # Remove trailing L from Unix timestamps
    created_clean = created_raw.str.replace(
        r"L$",
        "",
        regex=True
    )

    # ---------------------------------------------------------
    # 3. Create empty UTC datetime Series
    # ---------------------------------------------------------
    parsed_dates = pd.Series(
        pd.NaT,
        index=temp.index,
        dtype="datetime64[ns, UTC]"
    )

    # ---------------------------------------------------------
    # 4. Detect Unix millisecond timestamps
    # ---------------------------------------------------------
    is_unix_ms = created_clean.str.fullmatch(
        r"\d{10,15}"
    )

    # Parse Unix millisecond timestamps
    if is_unix_ms.any():

        parsed_dates.loc[is_unix_ms] = pd.to_datetime(
            pd.to_numeric(
                created_clean.loc[is_unix_ms],
                errors="coerce"
            ),
            unit="ms",
            errors="coerce",
            utc=True
        )

    # ---------------------------------------------------------
    # 5. Parse normal datetime strings
    # ---------------------------------------------------------
    normal_mask = ~is_unix_ms

    if normal_mask.any():

        parsed_dates.loc[normal_mask] = pd.to_datetime(
            created_clean.loc[normal_mask],
            errors="coerce",
            utc=True
        )

    # ---------------------------------------------------------
    # 6. Convert timezone-aware timestamps to naive UTC
    # ---------------------------------------------------------
    parsed_dates = parsed_dates.dt.tz_localize(None)

    temp["created_at"] = parsed_dates

    # ---------------------------------------------------------
    # 7. Remove invalid timestamps
    # ---------------------------------------------------------
    temp = temp.dropna(
        subset=["user_id", "created_at"]
    )

    # ---------------------------------------------------------
    # 8. Sort tweets chronologically for each account
    # ---------------------------------------------------------
    temp = temp.sort_values(
        ["user_id", "created_at"]
    )

    # ---------------------------------------------------------
    # 9. Calculate inter-tweet time
    # ---------------------------------------------------------
    temp["intertweet_seconds"] = (
        temp.groupby("user_id")["created_at"]
        .diff()
        .dt.total_seconds()
    )

    # ---------------------------------------------------------
    # 10. Inter-tweet statistics
    # ---------------------------------------------------------
    intertweet_stats = (
        temp.groupby("user_id")["intertweet_seconds"]
        .agg(
            mean_intertweet_seconds="mean",
            median_intertweet_seconds="median",
            std_intertweet_seconds="std"
        )
        .reset_index()
    )

    # ---------------------------------------------------------
    # 11. Inter-tweet coefficient of variation
    # ---------------------------------------------------------
    intertweet_stats["intertweet_cv"] = (
        intertweet_stats["std_intertweet_seconds"]
        /
        intertweet_stats["mean_intertweet_seconds"]
    )

    # ---------------------------------------------------------
    # 12. Burstiness
    # ---------------------------------------------------------
    intertweet_stats["burstiness"] = (
        (
            intertweet_stats["std_intertweet_seconds"]
            -
            intertweet_stats["mean_intertweet_seconds"]
        )
        /
        (
            intertweet_stats["std_intertweet_seconds"]
            +
            intertweet_stats["mean_intertweet_seconds"]
        )
    )

    # ---------------------------------------------------------
    # 13. Posting hour
    # ---------------------------------------------------------
    temp["posting_hour"] = (
        temp["created_at"].dt.hour
    )

    # ---------------------------------------------------------
    # 14. Posting-time entropy
    # ---------------------------------------------------------
    def calculate_hour_entropy(hours):

        counts = hours.value_counts(
            normalize=True
        )

        return -(
            counts * np.log2(counts)
        ).sum()

    hour_entropy = (
        temp.groupby("user_id")["posting_hour"]
        .apply(calculate_hour_entropy)
        .reset_index(
            name="posting_time_entropy"
        )
    )

    # ---------------------------------------------------------
    # 15. Number of active hours
    # ---------------------------------------------------------
    active_hour_count = (
        temp.groupby("user_id")["posting_hour"]
        .nunique()
        .reset_index(
            name="active_hour_count"
        )
    )

    # ---------------------------------------------------------
    # 16. Night activity ratio
    # 00:00 - 05:59
    # ---------------------------------------------------------
    temp["is_night"] = (
        temp["posting_hour"].between(0, 5)
    )

    night_activity_ratio = (
        temp.groupby("user_id")["is_night"]
        .mean()
        .reset_index(
            name="night_activity_ratio"
        )
    )

    # ---------------------------------------------------------
    # 17. Weekend activity ratio
    # Saturday = 5
    # Sunday = 6
    # ---------------------------------------------------------
    temp["weekday"] = (
        temp["created_at"].dt.weekday
    )

    temp["is_weekend"] = (
        temp["weekday"].isin([5, 6])
    )

    weekend_activity_ratio = (
        temp.groupby("user_id")["is_weekend"]
        .mean()
        .reset_index(
            name="weekend_activity_ratio"
        )
    )

    # ---------------------------------------------------------
    # 18. Top-hour concentration
    # ---------------------------------------------------------
    top_hour_concentration = (
        temp.groupby("user_id")["posting_hour"]
        .value_counts(normalize=True)
        .groupby(level=0)
        .max()
        .reset_index(
            name="top_hour_concentration"
        )
    )

    # ---------------------------------------------------------
    # 19. Daily tweet activity
    # ---------------------------------------------------------
    temp["date"] = (
        temp["created_at"].dt.date
    )

    daily_counts = (
        temp.groupby(
            ["user_id", "date"]
        )
        .size()
        .reset_index(
            name="daily_tweet_count"
        )
    )

    daily_activity_stats = (
        daily_counts.groupby("user_id")[
            "daily_tweet_count"
        ]
        .agg(
            mean_daily_tweets="mean",
            std_daily_tweets="std"
        )
        .reset_index()
    )

    # ---------------------------------------------------------
    # 20. Daily activity coefficient of variation
    # ---------------------------------------------------------
    daily_activity_stats["daily_activity_cv"] = (
        daily_activity_stats["std_daily_tweets"]
        /
        daily_activity_stats["mean_daily_tweets"]
    )

    # ---------------------------------------------------------
    # 21. Tweet count used
    # ---------------------------------------------------------
    tweet_count = (
        temp.groupby("user_id")
        .size()
        .reset_index(
            name="tweet_count"
        )
    )

    # ---------------------------------------------------------
    # 22. Build final temporal feature table
    # ---------------------------------------------------------
    temporal_features = intertweet_stats[
        [
            "user_id",
            "mean_intertweet_seconds",
            "median_intertweet_seconds",
            "intertweet_cv",
            "burstiness"
        ]
    ].copy()

    temporal_features = temporal_features.merge(
        hour_entropy,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        active_hour_count,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        night_activity_ratio,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        weekend_activity_ratio,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        top_hour_concentration,
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        daily_activity_stats[
            ["user_id", "daily_activity_cv"]
        ],
        on="user_id",
        how="left"
    )

    temporal_features = temporal_features.merge(
        tweet_count,
        on="user_id",
        how="left"
    )

    return temporal_features

In [39]:
dataset_paths = {
    "fake_followers":
        "../data/raw/datasets_full.csv/fake_followers.csv/fake_followers.csv/tweets.csv",

    "genuine_accounts":
        "../data/raw/datasets_full.csv/genuine_accounts.csv/genuine_accounts.csv/tweets.csv",

    "social_spambots_1":
        "../data/raw/datasets_full.csv/social_spambots_1.csv/social_spambots_1.csv/tweets.csv",

    "social_spambots_2":
        "../data/raw/datasets_full.csv/social_spambots_2.csv/social_spambots_2.csv/tweets.csv",

    "social_spambots_3":
        "../data/raw/datasets_full.csv/social_spambots_3.csv/social_spambots_3.csv/tweets.csv",

    "traditional_spambots_1":
        "../data/raw/datasets_full.csv/traditional_spambots_1.csv/traditional_spambots_1.csv/tweets.csv"
}

print("Datasets:", len(dataset_paths))

Datasets: 6


In [41]:
from pathlib import Path

base_path = Path("../data/raw/datasets_full.csv")

matches = list(base_path.rglob("genuine_accounts*"))

for p in matches:
    print(p)

..\data\raw\datasets_full.csv\genuine_accounts.csv


In [42]:
from pathlib import Path

genuine_path = Path("../data/raw/datasets_full.csv/genuine_accounts.csv")

print("Exists:", genuine_path.exists())
print("\nContents:")

for p in genuine_path.iterdir():
    print(p)

Exists: True

Contents:
..\data\raw\datasets_full.csv\genuine_accounts.csv\tweets.csv
..\data\raw\datasets_full.csv\genuine_accounts.csv\users.csv


In [43]:
from pathlib import Path

for account_type, path in dataset_paths.items():
    p = Path(path)
    print(
        f"{account_type:25} | "
        f"{'FOUND' if p.exists() else 'MISSING'} | "
        f"{p}"
    )

fake_followers            | FOUND | ..\data\raw\datasets_full.csv\fake_followers.csv\fake_followers.csv\tweets.csv
genuine_accounts          | MISSING | ..\data\raw\datasets_full.csv\genuine_accounts.csv\genuine_accounts.csv\tweets.csv
social_spambots_1         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_1.csv\social_spambots_1.csv\tweets.csv
social_spambots_2         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_2.csv\social_spambots_2.csv\tweets.csv
social_spambots_3         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_3.csv\social_spambots_3.csv\tweets.csv
traditional_spambots_1    | FOUND | ..\data\raw\datasets_full.csv\traditional_spambots_1.csv\traditional_spambots_1.csv\tweets.csv


In [44]:
dataset_paths = {
    "fake_followers":
        "../data/raw/datasets_full.csv/fake_followers.csv/fake_followers.csv/tweets.csv",

    "genuine_accounts":
        "../data/raw/datasets_full.csv/genuine_accounts.csv/tweets.csv",

    "social_spambots_1":
        "../data/raw/datasets_full.csv/social_spambots_1.csv/social_spambots_1.csv/tweets.csv",

    "social_spambots_2":
        "../data/raw/datasets_full.csv/social_spambots_2.csv/social_spambots_2.csv/tweets.csv",

    "social_spambots_3":
        "../data/raw/datasets_full.csv/social_spambots_3.csv/social_spambots_3.csv/tweets.csv",

    "traditional_spambots_1":
        "../data/raw/datasets_full.csv/traditional_spambots_1.csv/traditional_spambots_1.csv/tweets.csv"
}

In [45]:
for account_type, path in dataset_paths.items():
    p = Path(path)
    print(
        f"{account_type:25} | "
        f"{'FOUND' if p.exists() else 'MISSING'} | "
        f"{p}"
    )

fake_followers            | FOUND | ..\data\raw\datasets_full.csv\fake_followers.csv\fake_followers.csv\tweets.csv
genuine_accounts          | FOUND | ..\data\raw\datasets_full.csv\genuine_accounts.csv\tweets.csv
social_spambots_1         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_1.csv\social_spambots_1.csv\tweets.csv
social_spambots_2         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_2.csv\social_spambots_2.csv\tweets.csv
social_spambots_3         | FOUND | ..\data\raw\datasets_full.csv\social_spambots_3.csv\social_spambots_3.csv\tweets.csv
traditional_spambots_1    | FOUND | ..\data\raw\datasets_full.csv\traditional_spambots_1.csv\traditional_spambots_1.csv\tweets.csv


In [46]:
all_temporal_features = {}

for account_type, path in dataset_paths.items():
    print("\n" + "=" * 60)
    print("Processing:", account_type)

    tweets = pd.read_csv(
        path,
        encoding="latin1"
    )

    print("Tweets:", len(tweets))

    features = build_temporal_features(tweets)
    features["account_type"] = account_type

    all_temporal_features[account_type] = features

    print("Accounts:", len(features))
    print("Feature shape:", features.shape)

    del tweets


Processing: fake_followers


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(
C:\Users\namit\AppData\Local\Temp\ipykernel_1832\831449219.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Tweets: 196027
Accounts: 3202
Feature shape: (3202, 13)

Processing: genuine_accounts


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(


Tweets: 2839362


...

Tweets: 1418557


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\831449219.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Accounts: 464
Feature shape: (464, 13)

Processing: traditional_spambots_1
Tweets: 145094


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\831449219.py:10: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  temp["created_at"] = pd.to_datetime(


Accounts: 0
Feature shape: (0, 13)


In [47]:
temporal_dataset = pd.concat(
    all_temporal_features.values(),
    ignore_index=True
)

print("Final shape:", temporal_dataset.shape)

print("\nAccounts by category:")
print(
    temporal_dataset["account_type"]
    .value_counts()
)

Final shape: (9197, 13)

Accounts by category:
account_type
social_spambots_2    3457
fake_followers       3202
genuine_accounts     1083
social_spambots_1     991
social_spambots_3     464
Name: count, dtype: int64


In [48]:
temporal_dataset["label"] = (
    temporal_dataset["account_type"]
    != "genuine_accounts"
).astype(int)

In [49]:
print(
    temporal_dataset["label"]
    .value_counts()
)

label
1    8114
0    1083
Name: count, dtype: int64


In [50]:
print("Duplicate user IDs:",
      temporal_dataset["user_id"].duplicated().sum())

print("\nMissing values:")
print(
    temporal_dataset.isna().sum()
)

print("\nShape:")
print(temporal_dataset.shape)

Duplicate user IDs: 0

Missing values:
user_id                        0
mean_intertweet_seconds      114
median_intertweet_seconds    114
intertweet_cv                204
burstiness                   204
posting_time_entropy           0
active_hour_count              0
night_activity_ratio           0
weekend_activity_ratio         0
top_hour_concentration         0
daily_activity_cv            142
tweet_count                    0
account_type                   0
label                          0
dtype: int64

Shape:
(9197, 14)


In [51]:
trad_path = dataset_paths["traditional_spambots_1"]

trad_tweets = pd.read_csv(
    trad_path,
    encoding="latin1"
)

print("Shape:", trad_tweets.shape)
print("\ncreated_at sample:")
print(trad_tweets["created_at"].head(10))

print("\ncreated_at non-null:")
print(trad_tweets["created_at"].notna().sum())

print("\ncreated_at dtype:")
print(trad_tweets["created_at"].dtype)

Shape: (145094, 25)

created_at sample:
0    1283282654000L
1    1283282651000L
2    1283282592000L
3    1283282571000L
4    1283282543000L
5    1283282523000L
6    1283282509000L
7    1283282494000L
8    1282636240000L
9    1281125480000L
Name: created_at, dtype: str

created_at non-null:
145094

created_at dtype:
str


In [52]:
print(trad_tweets[["user_id", "created_at"]].head(20).to_string())

    user_id      created_at
0   7248952  1283282654000L
1   7248952  1283282651000L
2   7248952  1283282592000L
3   7248952  1283282571000L
4   7248952  1283282543000L
5   7248952  1283282523000L
6   7248952  1283282509000L
7   7248952  1283282494000L
8   7248952  1282636240000L
9   7248952  1281125480000L
10  7248952  1281124588000L
11  7248952  1281124587000L
12  7248952  1281124561000L
13  7248952  1281124545000L
14  7248952  1281124522000L
15  7248952  1281124482000L
16  7248952  1281124456000L
17  7248952  1281124350000L
18  7248952  1281124330000L
19  7248952  1281124015000L


In [54]:
trad_features = build_temporal_features(trad_tweets)

print("Accounts:", len(trad_features))
print("Shape:", trad_features.shape)
print("\nFirst rows:")
print(trad_features.head())

C:\Users\namit\AppData\Local\Temp\ipykernel_1832\176108972.py:12: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates = pd.to_datetime(


Accounts: 1000
Shape: (1000, 12)

First rows:
    user_id  mean_intertweet_seconds  median_intertweet_seconds  \
0   7248952             79276.919714                     1609.5   
1   7732472            134433.280884                    22997.0   
2   9524952             72479.873521                    21670.0   
3  10788822              7010.893092                       83.0   
4  14596967             24271.258599                     3521.0   

   intertweet_cv  burstiness  posting_time_entropy  active_hour_count  \
0      10.545250    0.826769              3.574847                 20   
1       9.655149    0.812297              3.802020                 24   
2       2.604003    0.445062              4.442701                 24   
3       5.790071    0.705452              4.463715                 24   
4       6.961493    0.748791              4.524466                 24   

   night_activity_ratio  weekend_activity_ratio  top_hour_concentration  \
0              0.176330              

In [59]:
all_temporal_features = {}

for account_type, path in dataset_paths.items():
    print("\n" + "=" * 60)
    print("Processing:", account_type)

    tweets = pd.read_csv(
        path,
        encoding="latin1"
    )

    print("Tweets:", len(tweets))

    features = build_temporal_features(tweets)
    features["account_type"] = account_type

    all_temporal_features[account_type] = features

    print("Accounts:", len(features))
    print("Feature shape:", features.shape)

    del tweets


Processing: fake_followers


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(
C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1676061094.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates.loc[normal_mask] = pd.to_datetime(


Tweets: 196027
Accounts: 3202
Feature shape: (3202, 13)

Processing: genuine_accounts


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: id) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(


Tweets: 2839362


...

Accounts: 3457
Feature shape: (3457, 13)

Processing: social_spambots_3


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1138504534.py:7: DtypeWarning: Columns (0: in_reply_to_screen_name, 1: place) have mixed types. Specify dtype option on import or set low_memory=False.
  tweets = pd.read_csv(


Tweets: 1418557


C:\Users\namit\AppData\Local\Temp\ipykernel_1832\1676061094.py:56: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  parsed_dates.loc[normal_mask] = pd.to_datetime(


Accounts: 464
Feature shape: (464, 13)

Processing: traditional_spambots_1
Tweets: 145094
Accounts: 1000
Feature shape: (1000, 13)


In [60]:
temporal_dataset = pd.concat(
    all_temporal_features.values(),
    ignore_index=True
)

print("Final temporal dataset shape:", temporal_dataset.shape)

print("\nAccounts by category:")
print(temporal_dataset["account_type"].value_counts())

print("\nDuplicate user IDs:",
      temporal_dataset["user_id"].duplicated().sum())

Final temporal dataset shape: (10197, 13)

Accounts by category:
account_type
social_spambots_2         3457
fake_followers            3202
genuine_accounts          1083
traditional_spambots_1    1000
social_spambots_1          991
social_spambots_3          464
Name: count, dtype: int64

Duplicate user IDs: 0


In [61]:
label_map = {
    "fake_followers": 1,
    "genuine_accounts": 0,
    "social_spambots_1": 1,
    "social_spambots_2": 1,
    "social_spambots_3": 1,
    "traditional_spambots_1": 1
}

temporal_dataset["label"] = (
    temporal_dataset["account_type"].map(label_map)
)

print(temporal_dataset.shape)
print(temporal_dataset["label"].value_counts())

(10197, 14)
label
1    9114
0    1083
Name: count, dtype: int64
